# COLCAP Correlation Analysis (Macro & Global Drivers)

Analyze whether a broad set of global macro and market variables help explain COLCAP (ICOLCAP) behavior.

**Macro rates & FX:** USDCOP · US10Y · DXY  
**Commodities:** Brent · WTI · Gold · Copper · Natural Gas  
**Equity indices:** S&P 500 · Nasdaq · Russell 2000 · EEM · ILF · Brazil (Bovespa) · Mexico (IPC) · Chile (IPSA)  
**Fixed income / credit:** TLT · LQD · HYG · EMB  
**Volatility:** VIX


In [ ]:
import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots


In [ ]:
# ── Professional Plotly theme ────────────────────────────────────────────────
import plotly.io as pio

NAVY    = '#1A3A5C'
RED     = '#C8102E'
PALETTE = ['#1A3A5C', '#2196F3', '#C8102E', '#2E7D32', '#6A0DAD', '#E67E00']

LAYOUT_BASE = dict(
    template='plotly_white',
    font=dict(family='Arial, sans-serif', size=12, color='#222222'),
    title_font=dict(size=15, color=NAVY, family='Arial, sans-serif'),
    paper_bgcolor='white',
    plot_bgcolor='#F8F9FA',
    margin=dict(l=60, r=40, t=70, b=60),
    colorway=PALETTE,
)


In [ ]:
SERIES_FILE = Path('../covariables_series.json')
DB_PATH = Path('../covariables_data.db')
START_DATE = '2018-01-01'
END_DATE = pd.Timestamp.today().date().isoformat()

---
## 1. Configuration

Set the data file paths and the date range of interest. All subsequent cells depend on these parameters.


In [ ]:
with SERIES_FILE.open() as f:
    series_cfg = json.load(f)['series']

friendly_to_ticker = dict(series_cfg)
ticker_to_friendly = {v: k for k, v in friendly_to_ticker.items()}
tickers = list(ticker_to_friendly.keys())


---
## 2. Universe & Data

Load the ticker universe from the JSON config file and pull historical prices from SQLite. Prices are pivoted to a wide format indexed by date, with one column per series.


In [ ]:
placeholders = ','.join('?' for _ in tickers)
sql = f'''
SELECT
    ticker,
    date,
    COALESCE(adj_close, close) AS price
FROM ticker_prices
WHERE ticker IN ({placeholders})
  AND date >= ?
  AND date <= ?
ORDER BY date
'''

with sqlite3.connect(DB_PATH) as conn:
    long_df = pd.read_sql_query(sql, conn, params=[*tickers, START_DATE, END_DATE])

long_df['date'] = pd.to_datetime(long_df['date'])
long_df['series'] = long_df['ticker'].map(ticker_to_friendly)

prices = (
    long_df
    .pivot(index='date', columns='series', values='price')
    .sort_index()
    .dropna(how='all')
)


In [ ]:
returns = prices.pct_change().dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()
corr = returns.corr()


---
## 3. Returns & Correlation Matrix

Compute daily percentage returns and the full Pearson correlation matrix across all series. The correlation matrix is the basis for the heatmap in Section 6.


In [ ]:
rolling_window = 60
rolling_corr = pd.DataFrame(index=returns.index)

for col in returns.columns:
    if col != 'COLCAP':
        rolling_corr[f'COLCAP_vs_{col}'] = returns['COLCAP'].rolling(rolling_window).corr(returns[col])


---
## 4. Rolling Correlations

A 60-day rolling Pearson correlation between COLCAP and each macro driver captures how the relationship shifts through time — useful for identifying regime changes (e.g., risk-off periods, commodity shocks, or rate-driven selloffs).


In [ ]:
target_name = 'COLCAP'
X_df = returns.drop(columns=[target_name]).copy()
y_sr = returns[target_name].copy()

model_df = pd.concat([y_sr, X_df], axis=1).dropna()
y = model_df[target_name].values
X = model_df.drop(columns=[target_name]).values

X_const = np.column_stack([np.ones(len(X)), X])
beta = np.linalg.lstsq(X_const, y, rcond=None)[0]
y_hat = X_const @ beta

ss_res = np.sum((y - y_hat) ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)
r2 = 1 - (ss_res / ss_tot)

coef = pd.Series(beta[1:], index=model_df.drop(columns=[target_name]).columns, name='coef')
intercept = beta[0]


---
## 5. Linear Explanatory Model (OLS)

Fit an OLS regression of COLCAP daily returns on the macro drivers to estimate how much of the index's day-to-day variation can be explained by these factors. The R² measures overall explanatory power; individual coefficients indicate the direction and magnitude of each driver's influence.


In [ ]:
# ── Correlation Heatmap ──────────────────────────────────────────────────────
labels = list(corr.columns)
z = corr.values.round(2)

# Custom text annotations with sign-aware colouring handled by zmin/zmax
text_vals = [[f'{v:.2f}' for v in row] for row in z]

fig = go.Figure(go.Heatmap(
    z=z,
    x=labels,
    y=labels,
    text=text_vals,
    texttemplate='%{text}',
    textfont=dict(size=12, family='Arial Black'),
    colorscale='RdBu',
    reversescale=True,   # red = negative, blue = positive
    zmid=0,
    zmin=-1,
    zmax=1,
    colorbar=dict(
        title=dict(text='Pearson r', side='right', font=dict(size=11)),
        tickvals=[-1, -0.5, 0, 0.5, 1],
        thickness=14,
        len=0.85,
    ),
))

fig.update_layout(
    **LAYOUT_BASE,
    title=dict(text='<b>Correlation Matrix — Daily Returns</b><br>'
                    '<sup>COLCAP &amp; Macro Drivers</sup>',
               x=0.5, xanchor='center'),
    width=700,
    height=580,
    xaxis=dict(tickangle=-40, side='bottom'),
    yaxis=dict(autorange='reversed'),
)

fig.show()


---
## 6. Visualizations

### 6.1 Correlation Heatmap
Static snapshot of pairwise Pearson correlations across the full sample period. Values close to ±1 indicate strong linear co-movement; values near 0 suggest independence.


In [ ]:
# ── Rolling Correlations: all macro drivers in one chart ─────────────────────
fig = go.Figure()

for idx, col in enumerate(rolling_corr.columns):
    label = col.replace('COLCAP_vs_', '')
    color = PALETTE[idx % len(PALETTE)]
    series = rolling_corr[col].dropna()

    fig.add_trace(go.Scatter(
        x=series.index,
        y=series.values,
        mode='lines',
        name=label,
        line=dict(color=color, width=2),
        hovertemplate=f'<b>{label}</b><br>%{{x|%b %d, %Y}}<br>Corr: %{{y:.3f}}<extra></extra>',
    ))

# Zero baseline
fig.add_hline(y=0, line=dict(color='#888888', width=1, dash='dot'))

# Shaded reference bands
for level, alpha in [(0.5, 0.05), (-0.5, 0.05)]:
    fig.add_hrect(
        y0=0, y1=level,
        fillcolor='#1A3A5C' if level > 0 else '#C8102E',
        opacity=0.04,
        layer='below',
        line_width=0,
    )

fig.update_layout(
    **LAYOUT_BASE,
    title=dict(
        text=f'<b>Rolling {rolling_window}-Day Correlation with COLCAP</b><br>'
             '<sup>USDCOP · Brent · S&amp;P 500 · VIX · US 10Y</sup>',
        x=0.5, xanchor='center',
    ),
    xaxis=dict(
        title='Date',
        showgrid=True,
        gridcolor='#E0E0E0',
        tickformat='%b %Y',
    ),
    yaxis=dict(
        title='Pearson Correlation',
        range=[-1.05, 1.05],
        tickvals=[-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1],
        tickformat='.2f',
        zeroline=False,
        showgrid=True,
        gridcolor='#E0E0E0',
    ),
    legend=dict(
        orientation='h',
        y=1.08,
        x=0.5,
        xanchor='center',
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='#CCCCCC',
        borderwidth=1,
    ),
    hovermode='x unified',
    width=950,
    height=480,
)

fig.show()


### 6.2 Rolling Correlations
Time-varying correlations reveal structural breaks and regime changes. Hover over the chart for exact values on any date.


In [ ]:
# ── OLS Coefficients Chart ───────────────────────────────────────────────────
sorted_coef = coef.sort_values()
bar_colors = [RED if v < 0 else NAVY for v in sorted_coef]

n_bars = len(sorted_coef)
chart_height = max(500, n_bars * 28 + 120)  # ~28px per bar, plus margins

ols_base = {k: v for k, v in LAYOUT_BASE.items() if k != 'margin'}

fig = go.Figure(go.Bar(
    x=sorted_coef.values,
    y=sorted_coef.index,
    orientation='h',
    marker=dict(color=bar_colors, opacity=0.88, line=dict(width=0)),
    text=[f'{v:+.4f}' for v in sorted_coef.values],
    textposition='outside',
    textfont=dict(size=11),
    cliponaxis=False,
))

fig.add_vline(x=0, line=dict(color='#444444', width=1.2))

x_range = max(abs(sorted_coef.min()), abs(sorted_coef.max())) * 1.5
fig.update_layout(
    **ols_base,
    title=dict(
        text=f'<b>OLS Coefficients — COLCAP Return Model</b><br>'
             f'<sup>R² = {r2:.4f} &nbsp;|&nbsp; Intercept = {intercept:+.6f}</sup>',
        x=0.5, xanchor='center',
    ),
    xaxis=dict(title='Coefficient', zeroline=False, range=[-x_range, x_range]),
    yaxis=dict(
        title='',
        tickfont=dict(size=11),
        automargin=True,
    ),
    width=800,
    height=chart_height,
    showlegend=False,
    margin=dict(l=120, r=80, t=70, b=50),
)

fig.show()


### 6.3 OLS Coefficient Chart
Each bar represents the marginal contribution of a macro driver to COLCAP daily returns, holding all others constant. The R² and intercept are annotated in the chart subtitle.


---
## 7. COLCAP vs Top 7 Short-Term Drivers

The seven series with the highest absolute Pearson correlation to COLCAP over the **most recent 60 trading days** — i.e., the drivers that matter *right now*, not on average over the full history. The legend shows both the 60-day and full-sample r values so you can spot regime shifts where a driver has become more or less relevant recently.


In [ ]:
# ── Top 7 drivers by recent (60-day) correlation with COLCAP ─────────────────
recent_prices      = prices.iloc[-(rolling_window + 1):]
recent_log_ret     = np.log(recent_prices / recent_prices.shift(1)).iloc[1:]
recent_corr_matrix = recent_log_ret.corr()

recent_corr = (
    recent_corr_matrix['COLCAP']
    .drop('COLCAP')
    .abs()
    .sort_values(ascending=False)
)
top7     = recent_corr.head(7).index.tolist()
selected = ['COLCAP'] + top7

ret_sel = recent_log_ret[selected]

start_lbl = ret_sel.index[0].strftime('%b %d, %Y')
end_lbl   = ret_sel.index[-1].strftime('%b %d, %Y')

# ── Z-score normalise so all series share one axis ───────────────────────────
z_sel = (ret_sel - ret_sel.mean()) / ret_sel.std()

# ── Single parallel chart ─────────────────────────────────────────────────────
fig = go.Figure()

# COLCAP — filled area, bold black
fig.add_trace(go.Scatter(
    x=z_sel.index,
    y=z_sel['COLCAP'],
    mode='lines+markers',
    name='COLCAP',
    line=dict(color='#000000', width=2.5),
    marker=dict(color='#000000', size=5, symbol='circle'),
    fill='tozeroy',
    fillcolor='rgba(0,0,0,0.06)',
    connectgaps=True,
    hovertemplate='<b>COLCAP</b><br>%{x|%b %d, %Y}<br>z = %{y:.2f}<extra></extra>',
))

for i, driver in enumerate(top7):
    r_recent = recent_corr_matrix.loc['COLCAP', driver]
    r_full   = corr.loc['COLCAP', driver]
    color    = PALETTE[i % len(PALETTE)]
    dash     = 'dot' if r_recent < 0 else 'solid'

    fig.add_trace(go.Scatter(
        x=z_sel.index,
        y=z_sel[driver],
        mode='lines+markers',
        name=f'{driver}  (60d r={r_recent:+.2f} | full r={r_full:+.2f})',
        line=dict(color=color, width=1.6, dash=dash),
        marker=dict(color=color, size=4, symbol='circle'),
        opacity=0.85,
        connectgaps=True,
        hovertemplate=f'<b>{driver}</b><br>%{{x|%b %d, %Y}}<br>z = %{{y:.2f}}<extra></extra>',
    ))

fig.add_hline(y=0, line=dict(color='#888888', width=1, dash='dot'))

fig.update_layout(
    **LAYOUT_BASE,
    title=dict(
        text=f'<b>COLCAP vs Top 7 Short-Term Drivers — Standardised Log Returns</b><br>'
             f'<sup>Last {rolling_window} trading days ({start_lbl} → {end_lbl}) · '
             f'z-score normalised · dotted line = negative 60-day correlation</sup>',
        x=0.5, xanchor='center',
    ),
    xaxis=dict(
        title='Date',
        showgrid=True, gridcolor='#E0E0E0',
        tickformat='%b %d',
    ),
    yaxis=dict(
        title='Standardised Return (σ)',
        showgrid=True, gridcolor='#E0E0E0',
        zeroline=False,
    ),
    legend=dict(
        orientation='h', y=-0.18, x=0.5, xanchor='center',
        bgcolor='rgba(255,255,255,0.85)', bordercolor='#CCCCCC', borderwidth=1,
        font=dict(size=10),
    ),
    hovermode='x unified',
    width=1000,
    height=480,
)

fig.show()


### 7.1 Current-Day Standardised Snapshot
The latest trading day inside the current 60-day window, shown as z-scores for COLCAP and its top short-term drivers.

In [ ]:
current_day_lbl = z_sel.index[-1].strftime('%b %d, %Y')

current_snapshot = pd.DataFrame({
    'series': z_sel.columns,
    'z_score': z_sel.iloc[-1].values,
    'log_return': ret_sel.iloc[-1].values,
    '60d_corr_vs_COLCAP': [1.0] + [recent_corr_matrix.loc['COLCAP', driver] for driver in top7],
    'full_corr_vs_COLCAP': [1.0] + [corr.loc['COLCAP', driver] for driver in top7],
})

current_snapshot['abs_z'] = current_snapshot['z_score'].abs()
current_snapshot = current_snapshot.sort_values('abs_z', ascending=False).drop(columns='abs_z')

color_map = {'COLCAP': '#000000'}
color_map.update({driver: PALETTE[i % len(PALETTE)] for i, driver in enumerate(top7)})

fig_current = go.Figure(go.Bar(
    x=current_snapshot['series'],
    y=current_snapshot['z_score'],
    marker=dict(color=current_snapshot['series'].map(color_map)),
    text=[f'{v:+.2f}' for v in current_snapshot['z_score']],
    textposition='outside',
    cliponaxis=False,
))
fig_current.add_hline(y=0, line=dict(color='#888888', width=1, dash='dot'))
fig_current.update_layout(
    **LAYOUT_BASE,
    title=dict(
        text=(
            f'<b>Current-Day Standardised Log Returns — COLCAP and Top Drivers</b><br>'
            f'<sup>{current_day_lbl} · top drivers ranked by recent 60-day correlation · '
            f'z-score normalised</sup>'
        ),
        x=0.5,
        xanchor='center',
    ),
    xaxis=dict(title='Series', tickangle=-35, showgrid=False),
    yaxis=dict(title='z-score', showgrid=True, gridcolor='#E0E0E0', zeroline=False),
    height=480,
    width=1000,
    showlegend=False,
    hovermode='closest',
)
fig_current.show()

current_snapshot


---
## 8. Actual Price Series — COLCAP & Top 7 Drivers

Raw price levels over the full sample, each series plotted against its own Y axis so different scales (e.g. index points vs. ETF price vs. rate) don't collapse any series to a flat line.


In [ ]:
import plotly.graph_objects as go

# ── Actual price series — COLCAP + top 4, each on its own Y axis ─────────────
plot_prices = prices[['COLCAP'] + top7].dropna(subset=['COLCAP']).copy()

# Main plot ends here; right-side axes live in a dedicated dock
x_domain_end = 0.74
axis_start = x_domain_end + 0.010
axis_gap = 0.038
right_positions = [axis_start + i * axis_gap for i in range(len(top7))]
yaxis_refs = ['y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8']

LABEL_X_OFFSET = 0.018  # shift label boxes this far to the right of each axis

def smart_tickformat(series_name: str) -> str:
    vmax = float(plot_prices[series_name].abs().max())
    return '.3s' if vmax >= 1000 else ',.1f'

# Extract anything already inside LAYOUT_BASE that would conflict later
base_layout = dict(LAYOUT_BASE)
base_annotations = list(base_layout.pop('annotations', []))
base_shapes = list(base_layout.pop('shapes', []))
base_margin = dict(base_layout.pop('margin', {}))

# COLCAP tick values — 6 evenly spaced across its full data range
colcap_data = plot_prices['COLCAP'].dropna()
colcap_min = float(colcap_data.min())
colcap_max = float(colcap_data.max())
colcap_tickvals = np.linspace(colcap_min, colcap_max, 6).tolist()

# Build the extra right-side axes
extra_axes = {}
axis_annotations = []

for i, driver in enumerate(top7):
    color = PALETTE[i % len(PALETTE)]
    axis_name = f'yaxis{i + 2}'

    d_data = plot_prices[driver].dropna()
    d_min = float(d_data.min())
    d_max = float(d_data.max())
    d_tickvals = np.linspace(d_min, d_max, 6).tolist()

    extra_axes[axis_name] = dict(
        anchor='free',
        overlaying='y',
        side='right',
        position=right_positions[i],

        showline=True,
        linecolor=color,
        linewidth=1.15,

        showgrid=False,
        zeroline=False,

        ticks='outside',
        ticklen=4,
        tickwidth=1,
        tickcolor=color,
        tickfont=dict(color=color, size=10),
        tickformat=smart_tickformat(driver),
        tickvals=d_tickvals,
        range=[d_min, d_max],
        separatethousands=True,

        showticklabels=True,
        title=dict(text=''),
    )

    # Colored label above each axis, shifted right
    axis_annotations.append(
        dict(
            x=right_positions[i] + LABEL_X_OFFSET,
            y=1.035,
            xref='paper',
            yref='paper',
            text=f'<b>{driver}</b>',
            showarrow=False,
            xanchor='center',
            yanchor='bottom',
            font=dict(size=10, color=color),
            bgcolor='rgba(255,255,255,0.92)',
            bordercolor=color,
            borderwidth=1,
            borderpad=2,
        )
    )

# Subtle background dock for the right-side axes
axis_dock_shapes = [
    dict(
        type='rect',
        xref='paper',
        yref='paper',
        x0=x_domain_end,
        x1=1.0,
        y0=0,
        y1=1,
        fillcolor='rgba(255,255,255,0.55)',
        line=dict(width=0),
        layer='below',
    ),
    dict(
        type='line',
        xref='paper',
        yref='paper',
        x0=x_domain_end,
        x1=x_domain_end,
        y0=0,
        y1=1,
        line=dict(color='rgba(120,120,120,0.35)', width=1),
        layer='above',
    ),
]

fig = go.Figure()

# COLCAP — primary left axis
fig.add_trace(
    go.Scatter(
        x=plot_prices.index,
        y=plot_prices['COLCAP'],
        mode='lines',
        name='COLCAP',
        yaxis='y',
        line=dict(color='#000000', width=2.7),
        connectgaps=True,
        hovertemplate='<b>COLCAP</b><br>%{x|%b %d, %Y}<br><b>%{y:,.2f}</b><extra></extra>',
    )
)

# Top 7 drivers — each on its own right-side axis
for i, driver in enumerate(top7):
    r_recent = recent_corr_matrix.loc['COLCAP', driver]
    r_full = corr.loc['COLCAP', driver]
    color = PALETTE[i % len(PALETTE)]
    dash = 'dot' if r_recent < 0 else 'solid'

    fig.add_trace(
        go.Scatter(
            x=plot_prices.index,
            y=plot_prices[driver],
            mode='lines',
            name=f'{driver}  (60d r={r_recent:+.2f} | full r={r_full:+.2f})',
            yaxis=yaxis_refs[i],
            line=dict(color=color, width=1.8, dash=dash),
            opacity=0.85,
            connectgaps=True,
            hovertemplate=f'<b>{driver}</b><br>%{{x|%b %d, %Y}}<br><b>%{{y:,.2f}}</b><extra></extra>',
        )
    )

final_layout = {
    **base_layout,
    **extra_axes,

    'title': dict(
        text=(
            f'<b>COLCAP vs Top 4 Drivers — Raw Price Series</b><br>'
            f'<sup>{START_DATE} → {END_DATE} · each series on its own Y axis · '
            f'dotted line = negative 60-day correlation</sup>'
        ),
        x=0.5,
        xanchor='center',
    ),

    'xaxis': dict(
        domain=[0.0, x_domain_end],
        title='Date',
        showgrid=True,
        gridcolor='#E0E0E0',
        tickformat='%b %Y',
        showline=True,
        linecolor='#B0B0B0',
        linewidth=1,
        zeroline=False,
    ),

    'yaxis': dict(
        title=dict(
            text='COLCAP',
            font=dict(color='#000000', size=12),
            standoff=8,
        ),
        tickfont=dict(color='#000000', size=10),
        tickformat='~s',
        tickvals=colcap_tickvals,
        range=[colcap_min, colcap_max],
        separatethousands=True,
        showline=True,
        linecolor='#000000',
        linewidth=1.3,
        showgrid=True,
        gridcolor='#E0E0E0',
        zeroline=False,
        ticks='outside',
        ticklen=4,
        tickwidth=1,
        tickcolor='#000000',
    ),

    'legend': dict(
        orientation='h',
        y=-0.18,
        x=0.5,
        xanchor='center',
        bgcolor='rgba(255,255,255,0.92)',
        bordercolor='#CCCCCC',
        borderwidth=1,
        font=dict(size=10),
    ),

    'annotations': base_annotations + axis_annotations,
    'shapes': base_shapes + axis_dock_shapes,

    'hovermode': 'closest',
    'width': 1400,
    'height': 560,
    'margin': {
        **base_margin,
        'l': 85,
        'r': 310,
        't': 95,
        'b': 110,
    },
}

fig.update_layout(**final_layout)
fig.show()


---
## 9. Actual Price Series — COLCAP & Top 7 Full-History Drivers

Same chart as Section 8, but the top 7 drivers are ranked by their absolute Pearson correlation with COLCAP over the **entire sample** (2018–2026) rather than just the most recent 60 trading days. This gives a structural, long-run view of which variables have consistently co-moved with COLCAP.

In [ ]:
# ── Top 7 by full-history absolute correlation ───────────────────────────────
top7_full = (
    corr['COLCAP']
    .drop('COLCAP')
    .abs()
    .sort_values(ascending=False)
    .head(7)
    .index.tolist()
)

plot_prices_full = prices[['COLCAP'] + top7_full].dropna(subset=['COLCAP']).copy()

x_domain_end_f = 0.74
axis_start_f   = x_domain_end_f + 0.010
axis_gap_f     = 0.038
right_positions_f = [axis_start_f + i * axis_gap_f for i in range(len(top7_full))]
yaxis_refs_f   = ['y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8']

def smart_tickformat_f(series_name):
    vmax = float(plot_prices_full[series_name].abs().max())
    return '.3s' if vmax >= 1000 else ',.1f'

base_layout_f       = dict(LAYOUT_BASE)
base_annotations_f  = list(base_layout_f.pop('annotations', []))
base_shapes_f       = list(base_layout_f.pop('shapes', []))
base_margin_f       = dict(base_layout_f.pop('margin', {}))

colcap_data_f    = plot_prices_full['COLCAP'].dropna()
colcap_min_f     = float(colcap_data_f.min())
colcap_max_f     = float(colcap_data_f.max())
colcap_tickvals_f = np.linspace(colcap_min_f, colcap_max_f, 6).tolist()

extra_axes_f      = {}
axis_annotations_f = []

for i, driver in enumerate(top7_full):
    color     = PALETTE[i % len(PALETTE)]
    axis_name = f'yaxis{i + 2}'

    d_data    = plot_prices_full[driver].dropna()
    d_min     = float(d_data.min())
    d_max     = float(d_data.max())
    d_tickvals = np.linspace(d_min, d_max, 6).tolist()

    extra_axes_f[axis_name] = dict(
        anchor='free',
        overlaying='y',
        side='right',
        position=right_positions_f[i],
        showline=True,  linecolor=color, linewidth=1.15,
        showgrid=False, zeroline=False,
        ticks='outside', ticklen=4, tickwidth=1, tickcolor=color,
        tickfont=dict(color=color, size=10),
        tickformat=smart_tickformat_f(driver),
        tickvals=d_tickvals,
        range=[d_min, d_max],
        separatethousands=True,
        showticklabels=True,
        title=dict(text=''),
    )

    axis_annotations_f.append(dict(
        x=right_positions_f[i] + LABEL_X_OFFSET,
        y=1.035,
        xref='paper', yref='paper',
        text=f'<b>{driver}</b>',
        showarrow=False,
        xanchor='center', yanchor='bottom',
        font=dict(size=10, color=color),
        bgcolor='rgba(255,255,255,0.92)',
        bordercolor=color, borderwidth=1, borderpad=2,
    ))

axis_dock_shapes_f = [
    dict(type='rect', xref='paper', yref='paper',
         x0=x_domain_end_f, x1=1.0, y0=0, y1=1,
         fillcolor='rgba(255,255,255,0.55)', line=dict(width=0), layer='below'),
    dict(type='line', xref='paper', yref='paper',
         x0=x_domain_end_f, x1=x_domain_end_f, y0=0, y1=1,
         line=dict(color='rgba(120,120,120,0.35)', width=1), layer='above'),
]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=plot_prices_full.index, y=plot_prices_full['COLCAP'],
    mode='lines', name='COLCAP', yaxis='y',
    line=dict(color='#000000', width=2.7),
    connectgaps=True,
    hovertemplate='<b>COLCAP</b><br>%{x|%b %d, %Y}<br><b>%{y:,.2f}</b><extra></extra>',
))

for i, driver in enumerate(top7_full):
    r_full   = corr.loc['COLCAP', driver]
    r_recent = recent_corr_matrix.loc['COLCAP', driver]
    color    = PALETTE[i % len(PALETTE)]
    dash     = 'dot' if r_full < 0 else 'solid'

    fig.add_trace(go.Scatter(
        x=plot_prices_full.index, y=plot_prices_full[driver],
        mode='lines',
        name=f'{driver}  (full r={r_full:+.2f} | 60d r={r_recent:+.2f})',
        yaxis=yaxis_refs_f[i],
        line=dict(color=color, width=1.8, dash=dash),
        opacity=0.85, connectgaps=True,
        hovertemplate=f'<b>{driver}</b><br>%{{x|%b %d, %Y}}<br><b>%{{y:,.2f}}</b><extra></extra>',
    ))

fig.update_layout(**{
    **base_layout_f,
    **extra_axes_f,
    'title': dict(
        text=(
            f'<b>COLCAP vs Top 7 Full-History Drivers — Raw Price Series</b><br>'
            f'<sup>{START_DATE} → {END_DATE} · each series on its own Y axis · '
            f'ranked by full-sample |r| · dotted line = negative full-history correlation</sup>'
        ),
        x=0.5, xanchor='center',
    ),
    'xaxis': dict(
        domain=[0.0, x_domain_end_f],
        title='Date',
        showgrid=True, gridcolor='#E0E0E0',
        tickformat='%b %Y',
        showline=True, linecolor='#B0B0B0', linewidth=1,
        zeroline=False,
    ),
    'yaxis': dict(
        title=dict(text='COLCAP', font=dict(color='#000000', size=12), standoff=8),
        tickfont=dict(color='#000000', size=10),
        tickformat='~s',
        tickvals=colcap_tickvals_f,
        range=[colcap_min_f, colcap_max_f],
        separatethousands=True,
        showline=True, linecolor='#000000', linewidth=1.3,
        showgrid=True, gridcolor='#E0E0E0',
        zeroline=False,
        ticks='outside', ticklen=4, tickwidth=1, tickcolor='#000000',
    ),
    'legend': dict(
        orientation='h', y=-0.18, x=0.5, xanchor='center',
        bgcolor='rgba(255,255,255,0.92)', bordercolor='#CCCCCC', borderwidth=1,
        font=dict(size=10),
    ),
    'annotations': base_annotations_f + axis_annotations_f,
    'shapes':      base_shapes_f      + axis_dock_shapes_f,
    'hovermode': 'closest',
    'width': 1400, 'height': 560,
    'margin': {**base_margin_f, 'l': 85, 'r': 310, 't': 95, 'b': 110},
})

fig.show()


---
## 10. Regime Analysis — Feb 9, 2022 → Oct 11, 2022

Focused window covering the lead-up to and aftermath of the Colombian presidential election (Gustavo Petro elected June 19, 2022) as well as the global rate-hike shock. This period is interesting because domestic political risk and global macro stress overlapped, potentially reshuffling which drivers dominated COLCAP.

The cell below computes, **for this window only**:
- Full correlation matrix heatmap
- Ranked bar chart of COLCAP pairwise correlations (all drivers)
- Z-scored price chart of COLCAP vs the top 7 drivers for that regime


In [ ]:

REGIME_START = '2022-02-09'
REGIME_END   = '2022-10-11'

# ── Slice prices and compute returns for the regime window ───────────────────
regime_prices  = prices.loc[REGIME_START:REGIME_END].copy()
# drop rows only when ALL series are NaN
regime_returns = regime_prices.pct_change().dropna(how='all')
# remove any column with no data at all in this window
regime_returns = regime_returns.dropna(axis=1, how='all')
# keep only rows where COLCAP itself has a valid return
regime_returns = regime_returns[regime_returns['COLCAP'].notna()]
regime_corr    = regime_returns.corr(min_periods=20)

# ── 1. Correlation Heatmap ───────────────────────────────────────────────────
r_labels   = list(regime_corr.columns)
r_z        = regime_corr.values.round(2)
r_textvals = [[f'{v:.2f}' for v in row] for row in r_z]

fig = go.Figure(go.Heatmap(
    z=r_z, x=r_labels, y=r_labels,
    text=r_textvals, texttemplate='%{text}',
    textfont=dict(size=11, family='Arial Black'),
    colorscale='RdBu', reversescale=True,
    zmid=0, zmin=-1, zmax=1,
    colorbar=dict(
        title=dict(text='Pearson r', side='right', font=dict(size=11)),
        tickvals=[-1, -0.5, 0, 0.5, 1],
        thickness=14, len=0.85,
    ),
))
fig.update_layout(
    **LAYOUT_BASE,
    title=dict(
        text=f'<b>Correlation Matrix — Daily Returns</b><br>'
             f'<sup>Regime window: {REGIME_START} → {REGIME_END}</sup>',
        x=0.5, xanchor='center',
    ),
    width=700, height=580,
    xaxis=dict(tickangle=-40, side='bottom'),
    yaxis=dict(autorange='reversed'),
)
fig.show()

# ── 2. COLCAP pairwise correlation bar chart ─────────────────────────────────
regime_colcap_corr = (
    regime_corr['COLCAP']
    .drop('COLCAP')
    .dropna()
    .sort_values()
)
r_bar_colors = [RED if v < 0 else NAVY for v in regime_colcap_corr]
r_ols_base   = {k: v for k, v in LAYOUT_BASE.items() if k != 'margin'}

fig2 = go.Figure(go.Bar(
    x=regime_colcap_corr.values,
    y=regime_colcap_corr.index,
    orientation='h',
    marker=dict(color=r_bar_colors, opacity=0.88, line=dict(width=0)),
    text=[f'{v:+.3f}' for v in regime_colcap_corr.values],
    textposition='outside',
    textfont=dict(size=11),
    cliponaxis=False,
))
fig2.add_vline(x=0, line=dict(color='#444444', width=1.2))

r_x_range = max(abs(regime_colcap_corr.min()), abs(regime_colcap_corr.max())) * 1.45
n_regime_bars = len(regime_colcap_corr)
fig2.update_layout(
    **r_ols_base,
    title=dict(
        text=f'<b>COLCAP Pairwise Correlations — Regime Window</b><br>'
             f'<sup>{REGIME_START} → {REGIME_END} · Pearson r on daily returns</sup>',
        x=0.5, xanchor='center',
    ),
    xaxis=dict(title='Pearson r', zeroline=False, range=[-r_x_range, r_x_range]),
    yaxis=dict(title='', tickfont=dict(size=11), automargin=True),
    width=800,
    height=max(500, n_regime_bars * 28 + 120),
    showlegend=False,
    margin=dict(l=120, r=80, t=70, b=50),
)
fig2.show()

# ── Top 7 drivers for regime ─────────────────────────────────────────────────
regime_top7 = (
    regime_corr['COLCAP']
    .drop('COLCAP')
    .dropna()
    .abs()
    .sort_values(ascending=False)
    .head(7)
    .index.tolist()
)

regime_sel = regime_returns[['COLCAP'] + regime_top7]

r_start_lbl = regime_sel.index[0].strftime('%b %d, %Y')
r_end_lbl   = regime_sel.index[-1].strftime('%b %d, %Y')

# ── 3. Cumulative returns: COLCAP vs top 7 drivers — rebased to 100 ──────────
regime_cumret = (1 + regime_sel).cumprod() * 100

fig3 = go.Figure()

# COLCAP — bold black, filled
fig3.add_trace(go.Scatter(
    x=regime_cumret.index, y=regime_cumret['COLCAP'],
    mode='lines', name='COLCAP',
    line=dict(color='#000000', width=2.8),
    fill='tozeroy', fillcolor='rgba(0,0,0,0.05)',
    connectgaps=True,
    hovertemplate='<b>COLCAP</b><br>%{x|%b %d, %Y}<br>Level: %{y:.1f} (start = 100)<extra></extra>',
))

for i, driver in enumerate(regime_top7):
    r_regime = regime_corr.loc['COLCAP', driver]
    r_full   = corr.loc['COLCAP', driver]
    color    = PALETTE[i % len(PALETTE)]
    dash     = 'dot' if r_regime < 0 else 'solid'

    fig3.add_trace(go.Scatter(
        x=regime_cumret.index, y=regime_cumret[driver],
        mode='lines',
        name=f'{driver}  (regime r={r_regime:+.2f} | full r={r_full:+.2f})',
        line=dict(color=color, width=1.8, dash=dash),
        opacity=0.88, connectgaps=True,
        hovertemplate=f'<b>{driver}</b><br>%{{x|%b %d, %Y}}<br>Level: %{{y:.1f}} (start = 100)<extra></extra>',
    ))

fig3.add_hline(y=100, line=dict(color='#888888', width=1, dash='dot'))

fig3.update_layout(
    **LAYOUT_BASE,
    title=dict(
        text=f'<b>COLCAP vs Top 7 Regime Drivers — Cumulative Return</b><br>'
             f'<sup>{r_start_lbl} → {r_end_lbl} · rebased to 100 at start · '
             f'dotted line = negative regime correlation</sup>',
        x=0.5, xanchor='center',
    ),
    xaxis=dict(title='Date', showgrid=True, gridcolor='#E0E0E0', tickformat='%b %Y'),
    yaxis=dict(
        title='Cumulative Return (start = 100)',
        showgrid=True, gridcolor='#E0E0E0',
        zeroline=False,
        tickformat=',.1f',
    ),
    legend=dict(
        orientation='h', y=-0.22, x=0.5, xanchor='center',
        bgcolor='rgba(255,255,255,0.85)', bordercolor='#CCCCCC', borderwidth=1,
        font=dict(size=10),
    ),
    hovermode='x unified',
    width=1000, height=500,
)
fig3.show()

# ── 4. Z-scored log returns: COLCAP vs top 7 drivers ────────────────────────
regime_log_ret = np.log(1 + regime_sel)
regime_z = (regime_log_ret - regime_log_ret.mean()) / regime_log_ret.std()

fig4 = go.Figure()

fig4.add_trace(go.Scatter(
    x=regime_z.index, y=regime_z['COLCAP'],
    mode='lines+markers', name='COLCAP',
    line=dict(color='#000000', width=2.5),
    marker=dict(color='#000000', size=4, symbol='circle'),
    fill='tozeroy', fillcolor='rgba(0,0,0,0.06)',
    connectgaps=True,
    hovertemplate='<b>COLCAP</b><br>%{x|%b %d, %Y}<br>z = %{y:.2f}<extra></extra>',
))

for i, driver in enumerate(regime_top7):
    r_regime = regime_corr.loc['COLCAP', driver]
    r_full   = corr.loc['COLCAP', driver]
    color    = PALETTE[i % len(PALETTE)]
    dash     = 'dot' if r_regime < 0 else 'solid'

    fig4.add_trace(go.Scatter(
        x=regime_z.index, y=regime_z[driver],
        mode='lines+markers',
        name=f'{driver}  (regime r={r_regime:+.2f} | full r={r_full:+.2f})',
        line=dict(color=color, width=1.6, dash=dash),
        marker=dict(color=color, size=3, symbol='circle'),
        opacity=0.85, connectgaps=True,
        hovertemplate=f'<b>{driver}</b><br>%{{x|%b %d, %Y}}<br>z = %{{y:.2f}}<extra></extra>',
    ))

fig4.add_hline(y=0, line=dict(color='#888888', width=1, dash='dot'))

fig4.update_layout(
    **LAYOUT_BASE,
    title=dict(
        text=f'<b>COLCAP vs Top 7 Regime Drivers — Standardised Log Returns</b><br>'
             f'<sup>{r_start_lbl} → {r_end_lbl} · z-score normalised · '
             f'dotted line = negative regime correlation</sup>',
        x=0.5, xanchor='center',
    ),
    xaxis=dict(title='Date', showgrid=True, gridcolor='#E0E0E0', tickformat='%b %Y'),
    yaxis=dict(
        title='Standardised Return (σ)',
        showgrid=True, gridcolor='#E0E0E0',
        zeroline=False,
    ),
    legend=dict(
        orientation='h', y=-0.22, x=0.5, xanchor='center',
        bgcolor='rgba(255,255,255,0.85)', bordercolor='#CCCCCC', borderwidth=1,
        font=dict(size=10),
    ),
    hovermode='x unified',
    width=1000, height=500,
)
fig4.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\nTop 7 drivers for COLCAP — regime {REGIME_START} → {REGIME_END}")
print(f"{'Driver':<20} {'Regime r':>10} {'Total return':>14} {'Full-sample r':>14}")
print("-" * 60)
for driver in ['COLCAP'] + regime_top7:
    r_regime    = regime_corr.loc['COLCAP', driver] if driver != 'COLCAP' else 1.0
    r_full      = corr.loc['COLCAP', driver] if driver != 'COLCAP' else 1.0
    total_ret   = (regime_cumret[driver].iloc[-1] - 100)
    r_str       = f'{r_regime:>+10.3f}' if driver != 'COLCAP' else '          —'
    rf_str      = f'{r_full:>+14.3f}'   if driver != 'COLCAP' else '             —'
    print(f"{driver:<20} {r_str} {total_ret:>+13.1f}% {rf_str}")


---
## 11. Regime Analysis — Feb 9, 2022 → Sep 22, 2022

Same methodology as §10 but with the window trimmed to September 22, 2022 — capturing the lead-up to the Petro inauguration (Aug 7) and the peak of the global rate-hike shock, before the October recovery.


In [ ]:

REGIME2_START = '2022-02-09'
REGIME2_END   = '2022-09-22'

# ── Slice prices and compute returns for the regime window ───────────────────
regime2_prices  = prices.loc[REGIME2_START:REGIME2_END].copy()
regime2_returns = regime2_prices.pct_change().dropna(how='all')
regime2_returns = regime2_returns.dropna(axis=1, how='all')
regime2_returns = regime2_returns[regime2_returns['COLCAP'].notna()]
regime2_corr    = regime2_returns.corr(min_periods=20)

# ── 1. Correlation Heatmap ───────────────────────────────────────────────────
r2_labels   = list(regime2_corr.columns)
r2_z        = regime2_corr.values.round(2)
r2_textvals = [[f'{v:.2f}' for v in row] for row in r2_z]

fig = go.Figure(go.Heatmap(
    z=r2_z, x=r2_labels, y=r2_labels,
    text=r2_textvals, texttemplate='%{text}',
    textfont=dict(size=11, family='Arial Black'),
    colorscale='RdBu', reversescale=True,
    zmid=0, zmin=-1, zmax=1,
    colorbar=dict(
        title=dict(text='Pearson r', side='right', font=dict(size=11)),
        tickvals=[-1, -0.5, 0, 0.5, 1],
        thickness=14, len=0.85,
    ),
))
fig.update_layout(
    **LAYOUT_BASE,
    title=dict(
        text=f'<b>Correlation Matrix — Daily Returns</b><br>'
             f'<sup>Regime window: {REGIME2_START} → {REGIME2_END}</sup>',
        x=0.5, xanchor='center',
    ),
    width=700, height=580,
    xaxis=dict(tickangle=-40, side='bottom'),
    yaxis=dict(autorange='reversed'),
)
fig.show()

# ── 2. COLCAP pairwise correlation bar chart ─────────────────────────────────
regime2_colcap_corr = (
    regime2_corr['COLCAP']
    .drop('COLCAP')
    .dropna()
    .sort_values()
)
r2_bar_colors = [RED if v < 0 else NAVY for v in regime2_colcap_corr]
r2_ols_base   = {k: v for k, v in LAYOUT_BASE.items() if k != 'margin'}

fig2 = go.Figure(go.Bar(
    x=regime2_colcap_corr.values,
    y=regime2_colcap_corr.index,
    orientation='h',
    marker=dict(color=r2_bar_colors, opacity=0.88, line=dict(width=0)),
    text=[f'{v:+.3f}' for v in regime2_colcap_corr.values],
    textposition='outside',
    textfont=dict(size=11),
    cliponaxis=False,
))
fig2.add_vline(x=0, line=dict(color='#444444', width=1.2))

r2_x_range = max(abs(regime2_colcap_corr.min()), abs(regime2_colcap_corr.max())) * 1.45
n2_regime_bars = len(regime2_colcap_corr)
fig2.update_layout(
    **r2_ols_base,
    title=dict(
        text=f'<b>COLCAP Pairwise Correlations — Regime Window</b><br>'
             f'<sup>{REGIME2_START} → {REGIME2_END} · Pearson r on daily returns</sup>',
        x=0.5, xanchor='center',
    ),
    xaxis=dict(title='Pearson r', zeroline=False, range=[-r2_x_range, r2_x_range]),
    yaxis=dict(title='', tickfont=dict(size=11), automargin=True),
    width=800,
    height=max(500, n2_regime_bars * 28 + 120),
    showlegend=False,
    margin=dict(l=120, r=80, t=70, b=50),
)
fig2.show()

# ── Top 7 drivers for regime ─────────────────────────────────────────────────
regime2_top7 = (
    regime2_corr['COLCAP']
    .drop('COLCAP')
    .dropna()
    .abs()
    .sort_values(ascending=False)
    .head(7)
    .index.tolist()
)

regime2_sel = regime2_returns[['COLCAP'] + regime2_top7]

r2_start_lbl = regime2_sel.index[0].strftime('%b %d, %Y')
r2_end_lbl   = regime2_sel.index[-1].strftime('%b %d, %Y')

# ── 3. Cumulative returns: COLCAP vs top 7 drivers — rebased to 100 ──────────
regime2_cumret = (1 + regime2_sel).cumprod() * 100

fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=regime2_cumret.index, y=regime2_cumret['COLCAP'],
    mode='lines', name='COLCAP',
    line=dict(color='#000000', width=2.8),
    fill='tozeroy', fillcolor='rgba(0,0,0,0.05)',
    connectgaps=True,
    hovertemplate='<b>COLCAP</b><br>%{x|%b %d, %Y}<br>Level: %{y:.1f} (start = 100)<extra></extra>',
))

for i, driver in enumerate(regime2_top7):
    r_regime2 = regime2_corr.loc['COLCAP', driver]
    r_full    = corr.loc['COLCAP', driver]
    color     = PALETTE[i % len(PALETTE)]
    dash      = 'dot' if r_regime2 < 0 else 'solid'

    fig3.add_trace(go.Scatter(
        x=regime2_cumret.index, y=regime2_cumret[driver],
        mode='lines',
        name=f'{driver}  (regime r={r_regime2:+.2f} | full r={r_full:+.2f})',
        line=dict(color=color, width=1.8, dash=dash),
        opacity=0.88, connectgaps=True,
        hovertemplate=f'<b>{driver}</b><br>%{{x|%b %d, %Y}}<br>Level: %{{y:.1f}} (start = 100)<extra></extra>',
    ))

fig3.add_hline(y=100, line=dict(color='#888888', width=1, dash='dot'))

fig3.update_layout(
    **LAYOUT_BASE,
    title=dict(
        text=f'<b>COLCAP vs Top 7 Regime Drivers — Cumulative Return</b><br>'
             f'<sup>{r2_start_lbl} → {r2_end_lbl} · rebased to 100 at start · '
             f'dotted line = negative regime correlation</sup>',
        x=0.5, xanchor='center',
    ),
    xaxis=dict(title='Date', showgrid=True, gridcolor='#E0E0E0', tickformat='%b %Y'),
    yaxis=dict(
        title='Cumulative Return (start = 100)',
        showgrid=True, gridcolor='#E0E0E0',
        zeroline=False,
        tickformat=',.1f',
    ),
    legend=dict(
        orientation='h', y=-0.22, x=0.5, xanchor='center',
        bgcolor='rgba(255,255,255,0.85)', bordercolor='#CCCCCC', borderwidth=1,
        font=dict(size=10),
    ),
    hovermode='x unified',
    width=1000, height=500,
)
fig3.show()

# ── 4. Z-scored log returns: COLCAP vs top 7 drivers ────────────────────────
regime2_log_ret = np.log(1 + regime2_sel)
regime2_z = (regime2_log_ret - regime2_log_ret.mean()) / regime2_log_ret.std()

fig4 = go.Figure()

fig4.add_trace(go.Scatter(
    x=regime2_z.index, y=regime2_z['COLCAP'],
    mode='lines+markers', name='COLCAP',
    line=dict(color='#000000', width=2.5),
    marker=dict(color='#000000', size=4, symbol='circle'),
    fill='tozeroy', fillcolor='rgba(0,0,0,0.06)',
    connectgaps=True,
    hovertemplate='<b>COLCAP</b><br>%{x|%b %d, %Y}<br>z = %{y:.2f}<extra></extra>',
))

for i, driver in enumerate(regime2_top7):
    r_regime2 = regime2_corr.loc['COLCAP', driver]
    r_full    = corr.loc['COLCAP', driver]
    color     = PALETTE[i % len(PALETTE)]
    dash      = 'dot' if r_regime2 < 0 else 'solid'

    fig4.add_trace(go.Scatter(
        x=regime2_z.index, y=regime2_z[driver],
        mode='lines+markers',
        name=f'{driver}  (regime r={r_regime2:+.2f} | full r={r_full:+.2f})',
        line=dict(color=color, width=1.6, dash=dash),
        marker=dict(color=color, size=3, symbol='circle'),
        opacity=0.85, connectgaps=True,
        hovertemplate=f'<b>{driver}</b><br>%{{x|%b %d, %Y}}<br>z = %{{y:.2f}}<extra></extra>',
    ))

fig4.add_hline(y=0, line=dict(color='#888888', width=1, dash='dot'))

fig4.update_layout(
    **LAYOUT_BASE,
    title=dict(
        text=f'<b>COLCAP vs Top 7 Regime Drivers — Standardised Log Returns</b><br>'
             f'<sup>{r2_start_lbl} → {r2_end_lbl} · z-score normalised · '
             f'dotted line = negative regime correlation</sup>',
        x=0.5, xanchor='center',
    ),
    xaxis=dict(title='Date', showgrid=True, gridcolor='#E0E0E0', tickformat='%b %Y'),
    yaxis=dict(
        title='Standardised Return (σ)',
        showgrid=True, gridcolor='#E0E0E0',
        zeroline=False,
    ),
    legend=dict(
        orientation='h', y=-0.22, x=0.5, xanchor='center',
        bgcolor='rgba(255,255,255,0.85)', bordercolor='#CCCCCC', borderwidth=1,
        font=dict(size=10),
    ),
    hovermode='x unified',
    width=1000, height=500,
)
fig4.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\nTop 7 drivers for COLCAP — regime {REGIME2_START} → {REGIME2_END}")
print(f"{'Driver':<20} {'Regime r':>10} {'Total return':>14} {'Full-sample r':>14}")
print("-" * 60)
for driver in ['COLCAP'] + regime2_top7:
    r_regime2   = regime2_corr.loc['COLCAP', driver] if driver != 'COLCAP' else 1.0
    r_full      = corr.loc['COLCAP', driver] if driver != 'COLCAP' else 1.0
    total_ret   = (regime2_cumret[driver].iloc[-1] - 100)
    r_str       = f'{r_regime2:>+10.3f}' if driver != 'COLCAP' else '          —'
    rf_str      = f'{r_full:>+14.3f}'   if driver != 'COLCAP' else '             —'
    print(f"{driver:<20} {r_str} {total_ret:>+13.1f}% {rf_str}")
